# DARF v6 — Hair-Specific Refinement

**v5 sonucu:** Composition + orientation + identity büyük ölçüde düzeldi, ama **Daenerys'in saç rengi (platinum/silver-white) hâlâ tam belli olmuyor**.

**Sebep:** `face_local_refine` yüz crop'una odaklı (bbox + 0.5–1.0 padding). Saç bbox'ın YUKARISINDA kaldığı için refine pass'inde tam denoise edilmiyor; orijinal full-image generation'da saçta hâlâ Hermione UNet leak'i var.

**v6 mekanizması: `hair_local_refine`** — yüz crop'unu **asimetrik olarak yukarı genişletir** (bbox h × 1.6 yukarı), saçı tamamen kapsar. Sadece o identity'nin LoRA'sı (α=1.6) aktif, prompt saç-odaklı (`platinum silver-white hair`, `bushy brown hair`).

Pipeline: **face_refine → hair_refine** (face geometry → hair texture).

## Bölüm 1 — Setup

In [ ]:
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
# BASE_MODEL = "RunDiffusion/Juggernaut-XL-v9"

In [ ]:
import os
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git {REPO_DIR}
else:
    print("Pulling...")
    !cd {REPO_DIR} && git pull

!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0" "mediapipe==0.10.14" "protobuf<5" scipy

os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  Base: {BASE_MODEL}")

## Bölüm 2 — Pipeline + Scorers + Identity overrides

Identity metadata'sında saç açıklamasını **birden fazla farklı şekilde** verelim — CLIP'in saç rengini birden çok kez "duyması" anchor güçlendirir.

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from attribute_scorer import AttributeScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display
import pipeline as P

identities = load_identities()

# === Saç açıklamaları çeşitlendirilmiş ve önde ===
identities["hermione"]["visual_description"] = (
    "front-facing young woman, long bushy brown curly hair, brunette, "
    "hazel eyes, fair skin, looking directly at the camera, black Hogwarts robe"
)
identities["daenerys"]["visual_description"] = (
    "front-facing woman, long straight platinum silver-white hair, icy blonde, "
    "violet eyes, pale skin, looking directly at the camera, regal blue dress"
)

# Saç phrase'lerini attention_phrases'in başına koy — hair_local_refine bunları kullanır
identities["hermione"]["attention_phrases"] = [
    "long bushy brown curly hair",
    "brunette hair",
    "Hermione_Granger",
    "black Hogwarts robe",
]
identities["daenerys"]["attention_phrases"] = [
    "long straight platinum silver-white hair",
    "icy blonde silver hair",
    "dae woman",
    "regal blue dress",
]

identities["hermione"]["negative_features"] = (
    "blonde, silver hair, platinum hair, white hair, icy hair, crown, "
    "side profile, turned head"
)
identities["daenerys"]["negative_features"] = (
    "brown hair, brunette, dark hair, bushy curly hair, witch robe, school uniform, "
    "side profile, turned head"
)

P._BASE_NEGATIVE = (
    "blurry, low quality, deformed face, cartoon, illustration, anime, 3d render, "
    "three women, three people, third person, extra person, woman in background, "
    "side profile, profile view, turned head, looking sideways"
)

pipe = build_pipeline(identities, base_model=BASE_MODEL)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
attr_scorer = AttributeScorer(identities, device="cuda")
evaluator   = Evaluator(device="cuda")
ref_embeds = {k: evaluator.extract_face_embedding(
    Image.open(identities[k]["reference_image"]).convert("RGB")) for k in identities}

names = list(identities.keys())
identity_regions = {names[0]: (0,0,512,1024), names[1]: (512,0,1024,1024)}

import numpy as np, cv2
def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))
def show_eval(label, image):
    sim_h, sim_d = evaluator.detect_and_assign_faces(image, ref_embeds["hermione"], ref_embeds["daenerys"])
    n = detected_face_count(image)
    print(f"[Eval/{label}] arc_h={sim_h:+.3f}  arc_d={sim_d:+.3f}  faces={n}")

print("Ready.")

## Bölüm 3 — Pose

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints
pose_dense = make_face_aware_pose(1024, 1024, front_facing=True, dense_face=True)
target_keypoints = get_face_aware_target_keypoints(front_facing=True)
display(pose_dense.resize((384, 384)))

## Bölüm 4 — Stage 1 + Stage 2 (v5 baseline)

In [ ]:
from pipeline import generate_two_stage

v6_two_stage = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.45},
        "daenerys": {"down": 0.15, "mid": 0.55, "up": 1.50},   # up'ı 1.4→1.5
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,
    refine_strength=0.35,
    refine_steps=28, layout_steps=28,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.20, "mid": 0.05, "up": 0.00},
    use_bg_lock=True, bg_lock_ratio=0.40, bg_lock_padding=24,
    bg_lock_in_layout=True, bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

stage1, stage2 = generate_two_stage(pipe, identities, pose_dense, seed=42, **v6_two_stage)
print("\n[Stage 1]");  display(stage1.resize((512,512)))
print("[Stage 2]");    display(stage2.resize((512,512)))
show_eval("v6_stage2", stage2)

## Bölüm 5 — Face Refine (yüz geometrisi düzeltme)

In [ ]:
from pipeline import face_local_refine

after_face = face_local_refine(
    pipe, identities, stage2,
    face_scorer, identity_regions,
    refine_strength=0.40,
    crop_pad_ratio=0.5,
    refine_steps=28,
    seed=42,
    feather=24,
)
print("\n[After face_refine]")
display(after_face.resize((512,512)))
show_eval("v6_after_face_refine", after_face)

## Bölüm 6 — **Hair Refine** (asıl yenilik)

Yüz crop'undan farklı:
- `up_pad=1.6` → bbox yüksekliğinin 1.6 katı YUKARI uzanır → saç tamamen dahil
- `horiz_pad=0.6` → yana orta genişleme
- `lora_alpha=1.6` → tek-LoRA agresif
- `refine_strength=0.55` → saç dokusunu gerçekten yeniden çizer
- Prompt sadece saç phrase'leri (`long straight platinum silver-white hair`)

In [ ]:
from pipeline import hair_local_refine

v6_final = hair_local_refine(
    pipe, identities, after_face,
    face_scorer, identity_regions,
    refine_strength=0.55,
    refine_steps=28,
    seed=42,
    feather=36,
    horiz_pad=0.6,
    up_pad=1.6,
    down_pad=0.2,
    lora_alpha=1.6,
)
print("\n[v6 FINAL — face_refine + hair_refine]")
display(v6_final.resize((512,512)))
show_eval("v6_final", v6_final)

## Bölüm 7 — Hair Refine Strength Sweep

Ne kadar saçı yeniden çizeyim sorusunun ablasyon eğrisi.

In [ ]:
hair_results = {}
for strength in [0.35, 0.45, 0.55, 0.65, 0.75]:
    print(f"\n=== hair_refine_strength={strength} ===")
    img = hair_local_refine(
        pipe, identities, after_face,
        face_scorer, identity_regions,
        refine_strength=strength,
        refine_steps=28, seed=42, feather=36,
        horiz_pad=0.6, up_pad=1.6, down_pad=0.2, lora_alpha=1.6,
    )
    hair_results[f"hair_s{strength:.2f}"] = img
    display(img.resize((448, 448)))
    show_eval(f"hair_s{strength:.2f}", img)

## Bölüm 8 — Hair Refine α Sweep

Tek-LoRA aktif, scale ne kadar olmalı?

In [ ]:
alpha_hair_results = {}
for a in [1.2, 1.4, 1.6, 1.8, 2.0]:
    print(f"\n=== hair_lora_alpha={a} ===")
    img = hair_local_refine(
        pipe, identities, after_face,
        face_scorer, identity_regions,
        refine_strength=0.55,
        refine_steps=28, seed=42, feather=36,
        horiz_pad=0.6, up_pad=1.6, down_pad=0.2, lora_alpha=a,
    )
    alpha_hair_results[f"hair_a{a:.1f}"] = img
    display(img.resize((448, 448)))
    show_eval(f"hair_a{a:.1f}", img)

## Bölüm 9 — Multi-Seed (rapor için final tablo)

In [ ]:
v6_seed_finals = {}
for s in [42, 123, 777]:
    print(f"\n=== v6 seed {s} ===")
    _, s2 = generate_two_stage(pipe, identities, pose_dense, seed=s, **v6_two_stage)
    after_f = face_local_refine(
        pipe, identities, s2, face_scorer, identity_regions,
        refine_strength=0.40, crop_pad_ratio=0.5, refine_steps=28, seed=s, feather=24,
    )
    final = hair_local_refine(
        pipe, identities, after_f, face_scorer, identity_regions,
        refine_strength=0.55, refine_steps=28, seed=s, feather=36,
        horiz_pad=0.6, up_pad=1.6, down_pad=0.2, lora_alpha=1.6,
    )
    v6_seed_finals[s] = final
    print("Stage 2:");        display(s2.resize((384, 384)))
    print("After face:");     display(after_f.resize((384, 384)))
    print("Final (+hair):");  display(final.resize((384, 384)))
    show_eval(f"v6_seed{s}_final", final)

## Bölüm 10 — Ablation + ZIP

In [ ]:
import pandas as pd
def collect(label, image):
    fc = face_scorer.score_image(image, identity_regions)
    pc = pose_scorer.score_image(image, target_keypoints, identity_regions)
    ac = attr_scorer.score_image(image, identity_regions)
    sim_h, sim_d = evaluator.detect_and_assign_faces(image, ref_embeds["hermione"], ref_embeds["daenerys"])
    n = detected_face_count(image)
    return [
        {"method": label, "identity": k,
         "arcface_self":  round(fc[k]["arcface"], 4),
         "dominance":     round(fc[k]["dominance"], 4),
         "pose":          round(pc[k]["pose"], 4),
         "attr_margin":   round(ac[k]["margin"], 4),
         "eval_arcface":  round(sim_h if k == "hermione" else sim_d, 4),
         "face_count":    n}
        for k in ["hermione", "daenerys"]
    ]

rows = []
if "stage2"      in dir(): rows += collect("v6_stage2",            stage2)
if "after_face"  in dir(): rows += collect("v6_after_face_refine", after_face)
if "v6_final"    in dir(): rows += collect("v6_final",             v6_final)
if "hair_results" in dir():
    for label, img in hair_results.items():
        rows += collect(label, img)
if "alpha_hair_results" in dir():
    for label, img in alpha_hair_results.items():
        rows += collect(label, img)
if "v6_seed_finals" in dir():
    for s, img in v6_seed_finals.items():
        rows += collect(f"v6_seed{s}_final", img)

df = pd.DataFrame(rows)
out = "data/results/darf_v6"
os.makedirs(out, exist_ok=True)
df.to_csv(f"{out}/ablation.csv", index=False)
df

In [ ]:
if "stage1"     in dir(): stage1.save(f"{out}/v6_stage1.png")
if "stage2"     in dir(): stage2.save(f"{out}/v6_stage2.png")
if "after_face" in dir(): after_face.save(f"{out}/v6_after_face_refine.png")
if "v6_final"   in dir(): v6_final.save(f"{out}/v6_final.png")
if "hair_results" in dir():
    for label, img in hair_results.items():
        img.save(f"{out}/{label}.png")
if "alpha_hair_results" in dir():
    for label, img in alpha_hair_results.items():
        img.save(f"{out}/{label}.png")
if "v6_seed_finals" in dir():
    for s, img in v6_seed_finals.items():
        img.save(f"{out}/v6_seed{s}_final.png")
pose_dense.save(f"{out}/pose_skeleton.png")

import shutil
from IPython.display import FileLink
shutil.make_archive("/workspace/darf_v6_results", "zip", out)
print("Zip ready: /workspace/darf_v6_results.zip")
FileLink("/workspace/darf_v6_results.zip")

## Akademik Çerçeve

v5 introduced block-anisotropic α scheduling to decouple identity from orientation, but residual hair-colour leakage persisted: face_local_refine restored facial geometry while leaving the hair region — which sits *outside* the face bounding box — untouched.

**v6 contribution: hair-region asymmetric refinement.**

We extend the post-processing stack with a third, hair-specific refinement pass that operates on an asymmetrically expanded crop (1.6× upward bbox extension to cover the full hair region) using only the target identity's LoRA at high α (1.6) and a hair-token-only prompt derived from `attention_phrases`. This decomposes identity refinement into two specialised stages:

$$\text{full image} \;\xrightarrow{\text{face\_refine}}\; \text{face geometry} \;\xrightarrow{\text{hair\_refine}}\; \text{hair texture/colour}$$

**Why it works:** rival LoRA hair leakage occurs above the face bbox in regions face_local_refine never touches; hair_local_refine isolates this region without disturbing facial features (low strength preserves the geometry restored by the previous stage; high strength would re-introduce orientation drift).

**Ablation (Bölüm 7-8):** strength and α sweeps chart the hair-quality trade-off curve; sweet spot at strength≈0.55, α≈1.6 in our setup.